In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.serif'] = ['Times New Roman']
rcParams['font.size'] = 10
rcParams['axes.labelsize'] = 10
rcParams['axes.titlesize'] = 12
rcParams['xtick.labelsize'] = 9
rcParams['ytick.labelsize'] = 9
rcParams['legend.fontsize'] = 9
rcParams['figure.titlesize'] = 14

# Get all CSV files in the current folder
csv_files = [f for f in os.listdir('.') if f.endswith('.csv') and 'intelligent' in f]
print(f"Found {len(csv_files)} data files: {csv_files}")

# Read and merge all the data
all_data = []
for file in csv_files:
    try:
        df = pd.read_csv(file)
        all_data.append(df)
        print(f"File read: {file}, containing {len(df)} rows of data")
    except Exception as e:
        print(f"An error occurred while reading the file {file}.: {e}")

if not all_data:
    print("No available data file was found.")
    exit()

#Merge all the data
combined_df = pd.concat(all_data, ignore_index=True)

# Remove the "smart_template" method
combined_df = combined_df[combined_df['method'] != 'smart_template']
print(f"\nThe size of the dataset after removing smart_template: {combined_df.shape}")
method_mapping = {
    'intelligent_SCL_method': 'SA-SCL',
    'miniCPM_only': 'miniCPM only',
    'markov_chain': 'Markov Chain',
    'cgan_method': 'CGAN',
    'twice': 'TWICE'
}

combined_df['method'] = combined_df['method'].map(method_mapping)
print(f"The remaining methods: {combined_df['method'].unique()}")

# Data cleaning and preprocessing
def clean_text(text):
    if isinstance(text, str):
        text = text.replace('```python', '').replace('```', '')
        text = text.replace('"""', '').replace('"', '')
        text = ' '.join(text.split())
    return text

combined_df['generated_post'] = combined_df['generated_post'].apply(clean_text)

# Define evaluation indicators
metrics = ['diversity', 'style_similarity', 'environment_relevance', 
           'habit_consistency', 'positivity_score', 'realness_score']
methods = combined_df['method'].unique()

print(f"\nThe methods included in the dataset: {list(methods)}")
print(f"Evaluation indicators: {metrics}")

# Calculate the statistical summaries for each method
method_stats = {}
for method in methods:
    method_data = combined_df[combined_df['method'] == method]
    stats = {}
    for metric in metrics:
        stats[metric] = {
            'mean': method_data[metric].mean(),
            'std': method_data[metric].std(),
            'min': method_data[metric].min(),
            'max': method_data[metric].max(),
            'median': method_data[metric].median(),
            'count': method_data[metric].count()
        }
    method_stats[method] = stats

# Create a summary DataFrame for visualization (including error information)
summary_data = []
for method in methods:
    for metric in metrics:
        mean_val = method_stats[method][metric]['mean']
        std_val = method_stats[method][metric]['std']
        summary_data.append({
            'Method': method,
            'Metric': metric,
            'Mean': mean_val,
            'Std': std_val,
            'Mean±Std': f"{mean_val:.4f}±{std_val:.4f}",
            'Min': method_stats[method][metric]['min'],
            'Max': method_stats[method][metric]['max']
        })
summary_df = pd.DataFrame(summary_data)



# Calculate the comprehensive score



# Calculate the weighted comprehensive score (the weight can be adjusted as needed) total==1
weights = { 
    'style_similarity': 0.167,
    'realness_score': 0.167,
    'habit_consistency': 0.167,
    'environment_relevance': 0.167,
    'diversity': 0.167,
    'positivity_score': 0.167
}

print(f"\nCalculation of weighted comprehensive score (Weight: {weights}):")
weighted_scores = {}
weighted_stds = {}

for method in methods:
    method_data = combined_df[combined_df['method'] == method]
    weighted_score = 0
    weighted_variance = 0
    
    for metric, weight in weights.items():
        mean_val = method_data[metric].mean()
        std_val = method_data[metric].std()
        weighted_score += mean_val * weight
        weighted_variance += (std_val * weight) ** 2
    
    weighted_std = np.sqrt(weighted_variance)
    weighted_scores[method] = weighted_score
    weighted_stds[method] = weighted_std


# Save the detailed results to Excel
output_path = 'method_comparison_results_no_smart_template.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # Original data
    combined_df.to_excel(writer, sheet_name='Original data', index=False)
    # Weight setting
    pd.DataFrame([weights]).to_excel(writer, sheet_name='Weight setting', index=False)




Found 29 data files: ['#AI_intelligent_results.csv', '#American_intelligent_results.csv', '#art_intelligent_results.csv', '#BBNaija_intelligent_results.csv', '#BillsMafia_intelligent_results.csv', '#Bitcoin_intelligent_results.csv', '#Covid19_intelligent_results.csv', '#Epstein_intelligent_results.csv', '#FC25_intelligent_results.csv', '#game_intelligent_results.csv', '#HanKuang_intelligent_results.csv', '#Hearts2Hearts_intelligent_results.csv', '#HongKong_intelligent_results.csv', '#Innovation_intelligent_results.csv', '#Japan_intelligent_results.csv', '#LALIGA_intelligent_results.csv', '#Mali_intelligent_results.csv', '#ML_intelligent_results.csv', '#Police_intelligent_results.csv', '#Russia_intelligent_results.csv', '#SaWWorldTourSG_intelligent_results.csv', '#SB19_intelligent_results.csv', '#SosCuba_intelligent_results.csv', '#technology_intelligent_results.csv', '#TikTok_intelligent_results.csv', '#US_intelligent_results.csv', '#Valorant_intelligent_results.csv', '#YNWA_intelligen